A pedagogical walkthrough using the Kaggle Give Me Some Credit dataset (~6.7% positive rate).

**Plan**

1. Load and inspect the data.
2. Stratified train/val split.
3. Baseline: logistic regression on the full imbalanced training set.
4. Comparison: logistic regression on a 1:1 downsampled training set.
5. Compare metrics (Average Precision, ROC-AUC, precision/recall/F1 at threshold, confusion matrix).
6. Compare predicted-score distributions on the validation set.

## 1. Load data

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv

# Registers and activates the shared `dvq` Plotly template.
import dvq_theme

# Load credentials from the repo-root .env, then map KAGGLE_API_TOKEN (KGAT_...) into KAGGLE_KEY for kagglehub.
load_dotenv(Path.cwd().parent.parent / ".env")
if os.environ.get("KAGGLE_API_TOKEN") and not os.environ.get("KAGGLE_KEY"):
    os.environ["KAGGLE_KEY"] = os.environ["KAGGLE_API_TOKEN"]

RANDOM_STATE = 42
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

In [2]:
import kagglehub

dataset_path = kagglehub.dataset_download("brycecf/give-me-some-credit-dataset")
csv_path = next(Path(dataset_path).glob("cs-training.csv"))
print(csv_path)

df = pd.read_csv(csv_path, index_col=0)
df = df.dropna(subset=["SeriousDlqin2yrs"])
df["SeriousDlqin2yrs"] = df["SeriousDlqin2yrs"].astype(int)

print(df.shape)
df["SeriousDlqin2yrs"].value_counts(normalize=True)

/Users/dvq/.cache/kagglehub/datasets/brycecf/give-me-some-credit-dataset/versions/1/cs-training.csv
(150000, 11)


SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64

## 2. Train/val split (stratified)

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["SeriousDlqin2yrs"])
y = df["SeriousDlqin2yrs"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"train: {len(X_train):>7,}   positives: {int(y_train.sum()):>5,}  ({y_train.mean()*100:.3f}%)")
print(f"val:   {len(X_val):>7,}   positives: {int(y_val.sum()):>5,}  ({y_val.mean()*100:.3f}%)")

train: 120,000   positives: 8,021  (6.684%)
val:    30,000   positives: 2,005  (6.683%)


## 3. Baseline: full imbalanced training set

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

baseline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
baseline.fit(X_train, y_train)

scores_baseline = baseline.predict_proba(X_val)[:, 1]

## 4. Comparison: 1:1 downsampled training set

In [5]:
pos_idx = y_train[y_train == 1].index
neg_idx = y_train[y_train == 0].index
neg_sampled = np.random.RandomState(RANDOM_STATE).choice(neg_idx, size=len(pos_idx), replace=False)
ds_idx = np.concatenate([pos_idx, neg_sampled])

X_train_ds = X_train.loc[ds_idx]
y_train_ds = y_train.loc[ds_idx]
print(f"downsampled train: {len(X_train_ds):,}  (positives: {int(y_train_ds.sum()):,})")

downsampled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
downsampled.fit(X_train_ds, y_train_ds)

scores_downsampled = downsampled.predict_proba(X_val)[:, 1]

downsampled train: 16,042  (positives: 8,021)


## 5. Metric comparison

In [6]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from IPython.display import display

def metrics_row(name, y_true, scores, threshold=0.5):
    y_pred = (scores >= threshold).astype(int)
    return {
        "model": name,
        "AP": average_precision_score(y_true, scores),
        "ROC-AUC": roc_auc_score(y_true, scores),
        f"precision@{threshold}": precision_score(y_true, y_pred, zero_division=0),
        f"recall@{threshold}": recall_score(y_true, y_pred, zero_division=0),
        f"f1@{threshold}": f1_score(y_true, y_pred, zero_division=0),
    }

metrics_df = pd.DataFrame([
    metrics_row("imbalanced (full)", y_val, scores_baseline),
    metrics_row("downsampled 1:1", y_val, scores_downsampled),
])
metrics_df.round(4).set_index("model")

,AP,ROC-AUC,precision@0.5,recall@0.5,f1@0.5
model,,,,,
imbalanced (full),0.2445,0.7142,0.5759,0.0454,0.0841
downsampled 1:1,0.3308,0.8068,0.1964,0.6564,0.3023


In [7]:
for name, scores in [("imbalanced (full)", scores_baseline), ("downsampled 1:1", scores_downsampled)]:
    y_pred = (scores >= 0.5).astype(int)
    cm = confusion_matrix(y_val, y_pred)
    print(f"{name} @ threshold=0.5")
    display(pd.DataFrame(cm, index=["actual 0", "actual 1"], columns=["pred 0", "pred 1"]))

imbalanced (full) @ threshold=0.5


,pred 0,pred 1
actual 0,27928,67
actual 1,1914,91


downsampled 1:1 @ threshold=0.5


,pred 0,pred 1
actual 0,22609,5386
actual 1,689,1316


The full data model has higher precision at threshold 0.5 (57.6% vs. 19.6% for the downsampled model) — but only because it barely makes any positive predictions. It flagged just 158 transactions out of 30,000 as likely defaults, catching 91 of the 2,005 actual defaults. That's 4.5% recall. When it fires, it's usually right; it almost never fires.

The downsampled model is the opposite: 1,315 defaults caught (65.6% recall) at the cost of 5,390 false alarms (19.6% precision). It's noisier but it actually tries to find defaults.

The problem with comparing models this way is that precision, recall, and F1 are all computed at a single threshold — 0.5 here. Move the threshold and every number changes. So instead of picking one threshold, what if we measured precision and recall at every possible threshold? Sweep the threshold from 1 down to 0, record precision and recall at each step, and plot them against each other — that's the precision-recall curve.

In [8]:
from sklearn.metrics import precision_recall_curve
from IPython.display import HTML

fig = go.Figure()

for name, scores, color in [
    ("imbalanced (full)", scores_baseline, "#999999"),
    ("downsampled 1:1", scores_downsampled, dvq_theme.ACCENT),
]:
    prec, rec, thr = precision_recall_curve(y_val, scores)
    # precision_recall_curve returns len(thr) == len(prec) - 1; pad to align
    thr_padded = np.concatenate([thr, [np.nan]])
    ap = average_precision_score(y_val, scores)
    fig.add_trace(go.Scatter(
        x=rec, y=prec,
        mode="lines",
        name=f"{name}  (AP={ap:.3f})",
        line=dict(color=color, width=2),
        customdata=thr_padded,
        hovertemplate="recall: %{x:.3f}<br>precision: %{y:.3f}<br>threshold: %{customdata:.3f}<extra></extra>",
    ))

random_precision = y_val.mean()
fig.add_hline(
    y=random_precision,
    line_dash="dot",
    line_color="#444444",
    annotation_text=f"random classifier ({random_precision:.3f})",
    annotation_position="bottom right",
    annotation_font_size=11,
)

fig.update_layout(
    title="Precision-Recall curve",
    autosize=True,
    height=420,
    xaxis=dict(title="Recall", range=[0, 1]),
    yaxis=dict(title="Precision", range=[0, 1]),
    legend=dict(x=0.45, y=0.95),
    hoverlabel=dict(bgcolor="#1a1a1a", font_color="white", bordercolor="#1a1a1a"),
)

HTML(fig.to_html(
    include_plotlyjs="cdn",
    full_html=False,
    div_id="fig-pr-curve",
    config={"responsive": True},
    default_width="100%",
    default_height="420px",
))

Each point on the curve is a threshold. Moving left to right, the threshold is dropping — you're flagging more borrowers as likely defaults (recall goes up), but you're also picking up more false alarms (precision goes down).^[Both curves show a brief ascending portion at the far left, where precision *increases* as recall increases. This happens because sklearn's `precision_recall_curve` computes precision and recall at each distinct score value, not one sample at a time. At the highest thresholds, you're only flagging a handful of predictions, and each time the threshold drops into a batch of samples that's majority positive, both recall and precision tick up together. The descent begins once you've exhausted that high-confidence positive cluster and start pulling in noisier, mixed-score territory.] The dotted line at 6.7% is the random classifier baseline. Picture a model whose scores carry no information about who actually defaults — noise, independent of the label (the exact distribution doesn't matter; uniform, Gaussian, anything works). Pick any threshold on such a model and the borrowers you flag are just a random sample of the population, so the fraction of them that actually defaults is simply the overall default rate: 6.7%. That holds at every recall level, no matter how many borrowers you flag, which is why the baseline is a flat line. It's the bar a real model has to clearly outperform to be worth anything.

A model with a higher curve is better at every operating point: for any recall target you pick, it achieves it at higher precision. The downsampled curve sits above the full data curve across almost the entire recall range, which confirms that the gap between these two models isn't a threshold artifact — it holds everywhere.

The area under this curve is what we want to maximize. It's a single number that summarizes the model's precision-recall tradeoff across all thresholds, without requiring you to commit to any one of them. Sklearn's `average_precision_score` computes this as the weighted mean of precision at each step, weighted by the change in recall — numerically equivalent to the area. That's **Average Precision (AP)**.

Finding a positive early in the ranked list contributes more to AP than finding one late — precision is higher near the top, so each early hit adds more area. This maps directly to how imbalanced problems actually get deployed: with a 6.7% default rate and a finite team of analysts, you'd flag roughly that top few percent of applications and never operate at a 50% flag rate. AP measures quality exactly in the region you'll actually use, making it often an important metric for imbalanced dataset.

That said, AP has one blind spot: it's sensitive to the positive rate in your evaluation set. In a monthly retraining setup, if the holdout happens to have fewer defaults than usual, AP drops even if the model's ranking quality is unchanged — not because the model got worse, but because there were fewer positives to find. To distinguish a real degradation from a base rate fluctuation, you need a metric that isn't affected by the positive rate at all. That's what the **ROC curve** provides.

The ROC curve comes from the same "sweep all thresholds" idea, but pairs recall with a different second axis. Instead of asking "of everyone I flagged, how many were real defaults?" (precision), it asks "of all safe borrowers, how many did I wrongly flag?" — that's the false positive rate (FPR). The reason this makes the ROC curve insensitive to the positive rate is that both axes are now computed on their own class in isolation. TPR only looks at actual defaulters, FPR only looks at actual safe borrowers, and neither calculation involves the other class. So the ratio of positives to negatives in your evaluation set never enters the picture. Sweep the threshold from 1 to 0, plot recall on the y-axis and FPR on the x-axis, and you get the ROC curve.

In [9]:
from sklearn.metrics import roc_curve

fig = go.Figure()

for name, scores, color in [
    ("imbalanced (full)", scores_baseline, "#999999"),
    ("downsampled 1:1", scores_downsampled, dvq_theme.ACCENT),
]:
    fpr, tpr, thr = roc_curve(y_val, scores)
    auc = roc_auc_score(y_val, scores)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode="lines",
        name=f"{name}  (AUC={auc:.3f})",
        line=dict(color=color, width=2),
        customdata=thr,
        hovertemplate="FPR: %{x:.3f}<br>TPR: %{y:.3f}<br>threshold: %{customdata:.3f}<extra></extra>",
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="random classifier",
    line=dict(color="#444444", width=1.5, dash="dot"),
    showlegend=True,
    hoverinfo="skip",
))

fig.update_layout(
    title="ROC curve",
    autosize=True,
    height=420,
    xaxis=dict(title="False Positive Rate", range=[0, 1]),
    yaxis=dict(title="True Positive Rate (Recall)", range=[0, 1]),
    legend=dict(x=0.45, y=0.12),
    hoverlabel=dict(bgcolor="#1a1a1a", font_color="white", bordercolor="#1a1a1a"),
)

HTML(fig.to_html(
    include_plotlyjs="cdn",
    full_html=False,
    div_id="fig-roc-curve",
    config={"responsive": True},
    default_width="100%",
    default_height="420px",
))

The area under the ROC curve is ROC-AUC. Picture the curve as a staircase you build by sweeping the threshold from high to low: each time you cross a non-defaulter, FPR ticks up and you take a horizontal step; each time you cross a defaulter, TPR ticks up and you take a vertical step. Area only accumulates on horizontal steps.

When you cross a non-defaulter, the rectangle you add has width 1/n_neg and height equal to the TPR at that moment — which is the fraction of defaulters already ranked above this non-defaulter. The area of that rectangle has a concrete interpretation: it counts how many defaulters beat this particular non-defaulter, normalized by the total number of (defaulter, non-defaulter) pairs. Sum those areas across all non-defaulters and you've counted every favorable pair (defaulter scored higher) exactly once. Here's that sweep animated with 2 defaulters (0.9, 0.5) and 3 non-defaulters (0.7, 0.3, 0.1) — hit Play and watch each threshold crossing:^[Pair-counting check on the same example: of the 6 possible (defaulter, non-defaulter) pairs, 5 have the defaulter scoring higher → area = 5/6 ≈ 0.833.]

In [10]:
import plotly.graph_objects as go
from IPython.display import HTML

# Tiny example from the footnote: 2 defaulters (D), 3 non-defaulters (N)
n_pos, n_neg = 2, 3
points = [(0.9, 'D'), (0.7, 'N'), (0.5, 'D'), (0.3, 'N'), (0.1, 'N')]

DIV_ID = 'fig-roc-auc-staircase-anim'


def _hex_rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    r, g, b = int(h[:2], 16), int(h[2:4], 16), int(h[4:], 16)
    return f'rgba({r},{g},{b},{alpha})'


ACCENT = dvq_theme.ACCENT
FILL_COLOR = _hex_rgba(ACCENT, 0.12)


def build_states():
    cx, cy = [0.0], [0.0]
    fpr, tpr, nps, nns, area = 0.0, 0.0, 0, 0, 0.0
    out = [{'cx': [0.0], 'cy': [0.0], 'area': 0.0, 'active': None,
            'label': 'Threshold above all scores — curve starts at origin'}]
    for i, (score, pt) in enumerate(points):
        if pt == 'D':
            nps += 1
            tpr = nps / n_pos
            cx.append(fpr); cy.append(tpr)
            label = (f'score={score} (Defaulter)<br>'
                     f'{nps}/{n_pos} defaulters seen → TPR = {tpr:.2f}  (vertical step, +0 area)')
        else:
            nns += 1
            old_fpr, fpr = fpr, nns / n_neg
            da = (fpr - old_fpr) * tpr
            area += da
            cx.append(fpr); cy.append(tpr)
            label = (f'score={score} (Non-defaulter)<br>'
                     f'{nns}/{n_neg} non-defaulters seen → FPR = {fpr:.2f}  (area +{da:.3f}, total {area:.3f})')
        out.append({'cx': list(cx), 'cy': list(cy), 'area': area, 'active': i, 'label': label})
    return out


states = build_states()


def fill_polygon(cx, cy):
    if len(cx) <= 1:
        return [0, 0, 0], [0, 0, 0]
    return cx + [cx[-1], 0], cy + [0, 0]


def score_strip_annots(active_idx):
    annots = []
    for i, (score, pt) in enumerate(points):
        if active_idx is None or i > active_idx:
            color, text = '#555555', f'{score}({pt})'
        elif i == active_idx:
            color = ACCENT if pt == 'D' else '#cccccc'
            text = f'<b>▶ {score}({pt})</b>'
        else:
            color = ACCENT if pt == 'D' else '#999999'
            text = f'{score}({pt})'
        annots.append(dict(
            x=(i + 0.5) / len(points), y=1.13,
            xref='paper', yref='paper',
            text=text, showarrow=False,
            font=dict(size=12, color=color),
        ))
    return annots


def step_annot(label):
    return dict(
        x=0.5, y=1.06, xref='paper', yref='paper',
        text=label, showarrow=False,
        font=dict(size=11.5), align='center',
    )


frames = []
for i, s in enumerate(states):
    fx, fy = fill_polygon(s['cx'], s['cy'])
    frames.append(go.Frame(
        name=str(i), traces=[1, 2],
        data=[
            go.Scatter(x=s['cx'], y=s['cy'], mode='lines',
                       line=dict(color=ACCENT, width=2.5)),
            go.Scatter(x=fx, y=fy, mode='none', fill='toself', fillcolor=FILL_COLOR),
        ],
        layout=go.Layout(
            annotations=score_strip_annots(s['active']) + [step_annot(s['label'])],
        ),
    ))

fx0, fy0 = fill_polygon(states[0]['cx'], states[0]['cy'])
fig = go.Figure(
    data=[
        go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                   line=dict(color='#555', dash='dot', width=1.5),
                   showlegend=False, hoverinfo='skip'),
        go.Scatter(x=states[0]['cx'], y=states[0]['cy'], mode='lines',
                   line=dict(color=ACCENT, width=2.5),
                   showlegend=False, hoverinfo='skip'),
        go.Scatter(x=fx0, y=fy0, mode='none', fill='toself', fillcolor=FILL_COLOR,
                   showlegend=False, hoverinfo='skip'),
    ],
    frames=frames,
    layout=go.Layout(
        height=460, autosize=True,
        margin=dict(t=125, b=90, l=60, r=40),
        xaxis=dict(title='False Positive Rate (FPR)', range=[-0.03, 1.03], zeroline=False),
        yaxis=dict(title='True Positive Rate (TPR)', range=[-0.03, 1.0], zeroline=False),
        annotations=score_strip_annots(None) + [step_annot(states[0]['label'])],
        updatemenus=[dict(
            type='buttons', showactive=False,
            x=0.5, y=-0.13, xanchor='center', yanchor='top',
            buttons=[
                dict(label='▶  Play', method='animate',
                     args=[None, {'frame': {'duration': 1400, 'redraw': True},
                                  'fromcurrent': True, 'mode': 'immediate',
                                  'transition': {'duration': 500}}]),
                dict(label='⏸  Pause', method='animate',
                     args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
            ],
        )],
        sliders=[dict(
            active=0,
            steps=[dict(
                method='animate',
                args=[[str(i)], {'mode': 'immediate',
                                  'frame': {'duration': 0, 'redraw': True},
                                  'transition': {'duration': 300}}],
                label=f'Step {i}',
            ) for i in range(len(states))],
            x=0.0, y=-0.04, len=1.0,
            currentvalue=dict(prefix='', xanchor='center', font=dict(size=11)),
            pad=dict(t=35),
        )],
        hoverlabel=dict(bgcolor='#1a1a1a', font_color='white'),
    ),
)

raw_html = fig.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    div_id=DIV_ID,
    config={'responsive': True},
    default_width='100%',
    default_height='560px',
)
# Plotly's to_html auto-plays all frames on load; replace with a jump to frame 0
# so the animation starts paused and waits for the reader to click Play.
raw_html = raw_html.replace(
    f"Plotly.animate('{DIV_ID}', null);",
    f"Plotly.animate('{DIV_ID}', ['0'], {{transition: {{duration: 0}}, frame: {{duration: 0, redraw: true}}}});",
)
HTML(raw_html)

That's why ROC-AUC equals **P(score(defaulter) > score(non-defaulter))** when you pick a random defaulter and a random non-defaulter from the dataset. The staircase is doing pair-counting in disguise: each non-defaulter "collects" the defaulters ranked above it, and the total area is favorable pairs divided by total pairs — exactly the probability for uniform sampling. A model that perfectly ranks all defaulters above all non-defaulters takes all its vertical steps before any horizontal ones, so every rectangle has height 1 — area = 1.0. Only the relative ordering of scores matters, not their actual values.


On the other hand, a random classifier ROC AUC baseline is exactly 0.5 — it ranks a positive above a negative half the time by chance, which is why it sits on the diagonal. The number has a direct probabilistic meaning regardless of class balance or score scale, and it lets you compare models fairly across datasets with different positive rates since neither TPR nor FPR depends on how many of the other class there are. That's why ROC-AUC is the default go-to metric for classification models.

In an imbalanced setting like ours, both models look considerably better in terms of ROC AUC than on the PR curve — AUC 0.807 vs AP 0.331 for the downsampled model. This is the FPR denominator effect: with 27,995 non-defaulters, even 1,400 false alarms register as just FPR = 0.05.

## 6. Score distributions on the validation set

Plotting each class separately with percentage on the y-axis keeps the two classes visually comparable, while hover tooltips surface the raw counts — so you can see both "what fraction of positives have a high score?" and "how many negatives is that tail actually?"

In [11]:
#| column: page
from IPython.display import HTML

bins = np.arange(0, 1.02, 0.02)
bin_centers = (bins[:-1] + bins[1:]) / 2
class_colors = {0: "#999999", 1: dvq_theme.ACCENT}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "negatives (class 0)", "positives (class 1)",
        "negatives (class 0)", "positives (class 1)",
    ),
    row_titles=["imbalanced (full)", "downsampled 1:1"],
    shared_yaxes="all",
    shared_xaxes="all",
    vertical_spacing=0.14,
    horizontal_spacing=0.1,
)

for row, (name, scores) in enumerate(
    [("imbalanced (full)", scores_baseline), ("downsampled 1:1", scores_downsampled)],
    start=1,
):
    for col, label in enumerate([0, 1], start=1):
        mask = y_val == label
        counts, _ = np.histogram(scores[mask], bins=bins)
        pct = counts / counts.sum() * 100
        fig.add_trace(
            go.Bar(
                x=bin_centers,
                y=pct,
                width=0.02,
                customdata=counts,
                hovertemplate="score: %{x:.2f}<br>%{y:.1f}%<br>count: %{customdata:,d}<extra></extra>",
                marker=dict(color=class_colors[label], line=dict(width=0)),
                showlegend=False,
            ),
            row=row, col=col,
        )

        n_above = int((scores[mask] >= 0.5).sum())
        fig.add_vline(
            x=0.5, row=row, col=col,
            line_dash="dash", line_color="#444444", line_width=1.5,
        )
        fig.add_annotation(
            x=0.52, y=88,
            text=f"{n_above:,} above 0.5",
            row=row, col=col,
            showarrow=False,
            xanchor="left",
            font=dict(size=11, color="#444444"),
        )

fig.update_layout(
    height=560,
    autosize=True,
    title="Score distributions by class — % within class (hover for raw count)",
    bargap=0,
    hoverlabel=dict(bgcolor="#1a1a1a", font_color="white", bordercolor="#1a1a1a"),
)
fig.update_xaxes(title_text="predicted P(class=1)", range=[0, 1])
fig.update_yaxes(title_text="% of class", range=[0, 100])

HTML(fig.to_html(
    include_plotlyjs="cdn",
    full_html=False,
    div_id="fig-scores-2x2",
    config={"responsive": True},
    default_width="100%",
    default_height="560px",
))

The full data model's positive side (top-right) is the striking panel: 95.5% of actual defaults score below 0.5. The model predicted "probably not a default" for nearly every borrower who ended up defaulting. The 91 true positives above 0.5 are a thin sliver of the 2,005 actual defaults.

The downsampled model shifts the positive distribution sharply upward. 65.6% of actual defaults now score above 0.5. But look at the negative side (bottom-left): 5,390 non-defaulters also crossed the threshold — about 19% of all safe borrowers. That's the cost of the shifted baseline.

The full data model avoided most of those false alarms precisely because its scores are anchored near zero. Raise the threshold above where both models agree there's no risk, and the full data model looks excellent. Drop it low enough that the downsampled model's inflation becomes relevant, and the story changes entirely. Neither model is categorically better — they're calibrated to different operating points.

## Why does training on downsampled data push scores higher?

Logistic regression models the probability of a positive outcome as:

$$p = \text{sigmoid}(w \cdot x + b) = \frac{1}{1 + e^{-(w \cdot x + b)}}$$

The weights $w$ learn which features push the probability up or down. The bias $b$ is a constant added to every sample's raw score before it gets squashed through sigmoid — the model's baseline suspicion before it looks at any features.

The model is trained to minimize cross-entropy loss:

$$L = -\sum_i \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right]$$

The $\log$ terms are what give this its character: $\log(p)$ shoots toward $-\infty$ as $p \to 0$, so being confidently wrong incurs a catastrophically large penalty. Think of it as a **"how surprised were you?" scorer**. Predict 99% default and it turns out to be a default — barely surprised, tiny penalty. Predict 1% default and it turns out to be a default — extremely surprised, huge penalty. The model's entire job during training is to stop being surprised.

Now zoom in on the bias. If it's set too low, the model is systematically shocked every time a default shows up. If it's too high, it's systematically shocked by the ~93% of borrowers who don't default. The only resting point where the bias stops accumulating surprise from both sides is when it matches how often defaults actually occur in training — the training positive rate. Any other value keeps bleeding into the loss.

This is why the downsampled model's scores are inflated. It was trained to stop being surprised in a world where defaults are 50% common. When deployed into a world where defaults are 6.7% common, its baseline suspicion is miscalibrated. Scores shift up. That's one effect.

But the metrics showed something more: the downsampled model also has a better ROC-AUC (0.807 vs 0.714). Score inflation alone can't explain that — ROC-AUC measures ranking, not scale. Something about the weights $w$ also changed.

The explanation is in how the gradient flows. With 93% negatives in the training set, 93% of the loss signal comes from the majority class. The model is hammered with "you predicted too high for non-defaulters" far more often than "you predicted too low for defaulters." Over many gradient steps, this asymmetry pushes the decision hyperplane toward a conservative location — one that separates non-defaulters cleanly without needing to find defaulters precisely.

With 1:1 downsampling, the gradient is symmetric. Both classes contribute equally to the loss, and the model learns weights that genuinely balance the trade-off. The result is a better separating hyperplane — not just a shifted score scale.

Both effects compound here. The weights improved and the bias shifted. In a dataset where the features are highly discriminative (or the model is expressive enough), gradient imbalance matters less — the model finds a good boundary regardless. Here, with 10 real features and a moderate class imbalance, the signal asymmetry is enough to meaningfully hurt LR's learned weights.

## What happens when we use `class_weight='balanced'`?

`class_weight='balanced'` keeps all the training data and instead multiplies each sample's contribution to the loss by a class-specific weight. Sklearn computes it as `n_samples / (n_classes × class_count)` — roughly 7.5× for defaults and 0.54× for non-defaulters in our dataset.

This is appealing because you don't throw away data. But the score calibration problem doesn't go away. With equal effective weighting, the bias settles at the same resting point as the downsampled model — governed by the effective class ratio in the loss, not the true base rate. Scores will still be inflated.

The genuine difference is in the feature weights $w$. The downsampled model trained on 16,042 examples; the balanced model trains on all 120k. More diverse non-defaulters should give the model a richer picture of the decision boundary. Whether that actually helps is an empirical question.

In [12]:
balanced = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")),
])
balanced.fit(X_train, y_train)

scores_balanced = balanced.predict_proba(X_val)[:, 1]

In [13]:
metrics_df = pd.DataFrame([
    metrics_row("imbalanced (full)", y_val, scores_baseline),
    metrics_row("downsampled 1:1", y_val, scores_downsampled),
    metrics_row("balanced weights", y_val, scores_balanced),
])
metrics_df.round(4).set_index("model")

,AP,ROC-AUC,precision@0.5,recall@0.5,f1@0.5
model,,,,,
imbalanced (full),0.2445,0.7142,0.5759,0.0454,0.0841
downsampled 1:1,0.3308,0.8068,0.1964,0.6564,0.3023
balanced weights,0.3235,0.8022,0.1817,0.6698,0.2859


In [14]:
for name, scores in [
    ("imbalanced (full)", scores_baseline),
    ("downsampled 1:1", scores_downsampled),
    ("balanced weights", scores_balanced),
]:
    y_pred = (scores >= 0.5).astype(int)
    cm = confusion_matrix(y_val, y_pred)
    print(f"{name} @ threshold=0.5")
    display(pd.DataFrame(cm, index=["actual 0", "actual 1"], columns=["pred 0", "pred 1"]))

imbalanced (full) @ threshold=0.5


,pred 0,pred 1
actual 0,27928,67
actual 1,1914,91


downsampled 1:1 @ threshold=0.5


,pred 0,pred 1
actual 0,22609,5386
actual 1,689,1316


balanced weights @ threshold=0.5


,pred 0,pred 1
actual 0,21948,6047
actual 1,662,1343


In [15]:
#| column: page
bins = np.arange(0, 1.02, 0.02)
bin_centers = (bins[:-1] + bins[1:]) / 2
class_colors = {0: "#999999", 1: dvq_theme.ACCENT}

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        "negatives (class 0)", "positives (class 1)",
        "negatives (class 0)", "positives (class 1)",
        "negatives (class 0)", "positives (class 1)",
    ),
    row_titles=["imbalanced (full)", "downsampled 1:1", "balanced weights"],
    shared_yaxes="all",
    shared_xaxes="all",
    vertical_spacing=0.10,
    horizontal_spacing=0.1,
)

for row, (name, scores) in enumerate(
    [
        ("imbalanced (full)", scores_baseline),
        ("downsampled 1:1", scores_downsampled),
        ("balanced weights", scores_balanced),
    ],
    start=1,
):
    for col, label in enumerate([0, 1], start=1):
        mask = y_val == label
        counts, _ = np.histogram(scores[mask], bins=bins)
        pct = counts / counts.sum() * 100
        fig.add_trace(
            go.Bar(
                x=bin_centers,
                y=pct,
                width=0.02,
                customdata=counts,
                hovertemplate="score: %{x:.2f}<br>%{y:.1f}%<br>count: %{customdata:,d}<extra></extra>",
                marker=dict(color=class_colors[label], line=dict(width=0)),
                showlegend=False,
            ),
            row=row, col=col,
        )

        n_above = int((scores[mask] >= 0.5).sum())
        fig.add_vline(
            x=0.5, row=row, col=col,
            line_dash="dash", line_color="#444444", line_width=1.5,
        )
        fig.add_annotation(
            x=0.52, y=88,
            text=f"{n_above:,} above 0.5",
            row=row, col=col,
            showarrow=False,
            xanchor="left",
            font=dict(size=11, color="#444444"),
        )

fig.update_layout(
    height=800,
    autosize=True,
    title="Score distributions by class — % within class (hover for raw count)",
    bargap=0,
    hoverlabel=dict(bgcolor="#1a1a1a", font_color="white", bordercolor="#1a1a1a"),
)
fig.update_xaxes(title_text="predicted P(class=1)", range=[0, 1])
fig.update_yaxes(title_text="% of class", range=[0, 100])

HTML(fig.to_html(
    include_plotlyjs="cdn",
    full_html=False,
    div_id="fig-scores-3x2",
    config={"responsive": True},
    default_width="100%",
    default_height="800px",
))

The metrics table:

| model | AP | ROC-AUC |
|---|---|---|
| imbalanced (full) | 0.245 | 0.714 |
| balanced weights | 0.323 | 0.802 |
| downsampled 1:1 | **0.331** | **0.807** |

Both downsampling and balanced weights outperform the full data model on every ranking metric — which is itself the finding worth dwelling on. The usual expectation is that more data wins. Here it doesn't, at least for LR.

Balanced weights (AP 0.323) ends up just below downsampled (0.331), despite training on 7.5× more examples. The 120k training set with weighted loss didn't extract meaningfully more signal than the 16k balanced subset. The score calibration is equally broken for both — both were optimized toward an effective 50/50 world — and the richer diversity of non-defaulters in the weighted model only moved the needle by 0.008 AP.

The false positive counts at threshold 0.5 tell the same story as before: the full data model sends 67 false alarms and catches 91 defaults; the balanced model sends 6,051 false alarms and catches 1,342 defaults. Same trade-off structure as downsampling — precision vs recall, not better ranking overall.

The one thing neither approach solves is calibration. To get scores that mean something as probabilities — where 0.7 actually means 70% chance of default — you need an explicit calibration step on top.

## LightGBM: does the same story hold?

LightGBM's equivalent of `class_weight='balanced'` is `scale_pos_weight`. Setting it to `n_neg / n_pos` (~14 here) makes the total loss contribution from positives equal to negatives — same math, different API.

One important difference from logistic regression: LightGBM needs explicit early stopping. Logistic regression optimizes log-loss globally and the gradient naturally accounts for every example throughout training. LightGBM builds trees sequentially, and each tree can overfit the residuals of the previous ones — particularly when the minority class is sparse relative to the feature space. The point where additional trees start hurting AP on held-out data is non-deterministic across runs due to parallel thread scheduling.

The fix is early stopping: carve out 20% of training data as a validation set, track AP directly, and stop when it stops improving.

In [16]:
import lightgbm as lgb

n_pos = int(y_train.sum())
n_neg = int((y_train == 0).sum())

# Carve out 20% of training data for early stopping (stratified so it has ~79 positives).
# X_val is kept fully held-out for final evaluation.
X_tr, X_es, y_tr, y_es = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)
print(f"LGBM training set:       {len(X_tr):,}  ({int(y_tr.sum())} positives)")
print(f"Early stopping set:      {len(X_es):,}  ({int(y_es.sum())} positives)")

def ap_metric(y_true, y_pred):
    return "ap", average_precision_score(y_true, y_pred), True  # higher is better

callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)]

def fit_lgbm(X_fit, y_fit, **kwargs):
    m = lgb.LGBMClassifier(
        n_estimators=2000, random_state=RANDOM_STATE, verbose=-1,
        num_threads=1,  # deterministic: parallel threading changes split order
        **kwargs
    )
    m.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric=ap_metric, callbacks=callbacks)
    return m

lgbm_baseline    = fit_lgbm(X_tr, y_tr)
lgbm_balanced    = fit_lgbm(X_tr, y_tr, scale_pos_weight=n_neg / n_pos)
lgbm_downsampled = fit_lgbm(X_train_ds, y_train_ds)

scores_lgbm_baseline    = lgbm_baseline.predict_proba(X_val)[:, 1]
scores_lgbm_balanced    = lgbm_balanced.predict_proba(X_val)[:, 1]
scores_lgbm_downsampled = lgbm_downsampled.predict_proba(X_val)[:, 1]

print(f"\nscale_pos_weight: {n_neg / n_pos:.1f}×")
for name, m in [("baseline", lgbm_baseline), ("scale_pos_weight", lgbm_balanced), ("downsampled", lgbm_downsampled)]:
    print(f"  LGBM {name:18s}  best_iter={m.best_iteration_}")

LGBM training set:       96,000  (6417 positives)
Early stopping set:      24,000  (1604 positives)



scale_pos_weight: 14.0×
  LGBM baseline            best_iter=41
  LGBM scale_pos_weight    best_iter=1
  LGBM downsampled         best_iter=54


In [17]:
metrics_all = pd.DataFrame([
    metrics_row("LR — imbalanced (full)", y_val, scores_baseline),
    metrics_row("LR — downsampled 1:1", y_val, scores_downsampled),
    metrics_row("LR — balanced weights", y_val, scores_balanced),
    metrics_row("LGBM — imbalanced (full)", y_val, scores_lgbm_baseline),
    metrics_row("LGBM — downsampled 1:1", y_val, scores_lgbm_downsampled),
    metrics_row("LGBM — balanced weights", y_val, scores_lgbm_balanced),
])
metrics_all.round(4).set_index("model")

,AP,ROC-AUC,precision@0.5,recall@0.5,f1@0.5
model,,,,,
LR — imbalanced (full),0.2445,0.7142,0.5759,0.0454,0.0841
LR — downsampled 1:1,0.3308,0.8068,0.1964,0.6564,0.3023
LR — balanced weights,0.3235,0.8022,0.1817,0.6698,0.2859
LGBM — imbalanced (full),0.4057,0.8679,0.6280,0.1810,0.2811
LGBM — downsampled 1:1,0.3978,0.8673,0.2122,0.7776,0.3334
LGBM — balanced weights,0.3276,0.8553,0.0000,0.0000,0.0000


In [18]:
#| column: page
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        "negatives (class 0)", "positives (class 1)",
        "negatives (class 0)", "positives (class 1)",
        "negatives (class 0)", "positives (class 1)",
    ),
    row_titles=["LGBM — imbalanced (full)", "LGBM — downsampled 1:1", "LGBM — balanced weights"],
    shared_yaxes="all",
    shared_xaxes="all",
    vertical_spacing=0.10,
    horizontal_spacing=0.1,
)

for row, (name, scores) in enumerate(
    [
        ("LGBM — imbalanced (full)", scores_lgbm_baseline),
        ("LGBM — downsampled 1:1", scores_lgbm_downsampled),
        ("LGBM — balanced weights", scores_lgbm_balanced),
    ],
    start=1,
):
    for col, label in enumerate([0, 1], start=1):
        mask = y_val == label
        counts, _ = np.histogram(scores[mask], bins=bins)
        pct = counts / counts.sum() * 100
        fig.add_trace(
            go.Bar(
                x=bin_centers,
                y=pct,
                width=0.02,
                customdata=counts,
                hovertemplate="score: %{x:.2f}<br>%{y:.1f}%<br>count: %{customdata:,d}<extra></extra>",
                marker=dict(color=class_colors[label], line=dict(width=0)),
                showlegend=False,
            ),
            row=row, col=col,
        )

        n_above = int((scores[mask] >= 0.5).sum())
        fig.add_vline(
            x=0.5, row=row, col=col,
            line_dash="dash", line_color="#444444", line_width=1.5,
        )
        fig.add_annotation(
            x=0.52, y=88,
            text=f"{n_above:,} above 0.5",
            row=row, col=col,
            showarrow=False,
            xanchor="left",
            font=dict(size=11, color="#444444"),
        )

fig.update_layout(
    height=800,
    autosize=True,
    title="LightGBM score distributions by class — % within class (hover for raw count)",
    bargap=0,
    hoverlabel=dict(bgcolor="#1a1a1a", font_color="white", bordercolor="#1a1a1a"),
)
fig.update_xaxes(title_text="predicted P(class=1)", range=[0, 1])
fig.update_yaxes(title_text="% of class", range=[0, 100])

HTML(fig.to_html(
    include_plotlyjs="cdn",
    full_html=False,
    div_id="fig-scores-lgbm-3x2",
    config={"responsive": True},
    default_width="100%",
    default_height="800px",
))

For LGBM, the story flips back to the expected direction — full data wins AP.

| model | AP | ROC-AUC | best_iter |
|---|---|---|---|
| LR — imbalanced (full) | 0.245 | 0.714 | — |
| LR — balanced weights | 0.323 | 0.802 | — |
| LR — downsampled 1:1 | 0.331 | **0.807** | — |
| LGBM — imbalanced (full) | **0.406** | **0.868** | 49 |
| LGBM — downsampled 1:1 | 0.398 | 0.868 | 45 |
| LGBM — scale_pos_weight | 0.326 | 0.855 | 1 |

LGBM full data (AP 0.406) beats downsampled (0.398) and scaled weighting (0.326). The margin between full and downsampled is slim — 0.008 AP — but the direction is what matters. `best_iter` of 49 and 45 are both realistic: the model needed real gradient accumulation to find the signal in these features, unlike the single-tree result on the PCA-preprocessed credit card fraud dataset.

`scale_pos_weight=14` crashes out at iter 1 with all predictions below 0.5 (precision@0.5 = 0.0). Even with 14× weighting, the first tree's aggressive split toward the minority class tanks the AP on the early stopping set immediately — subsequent trees only make it worse.

The contrast between LR and LGBM is where the actual lesson lives. LR, a linear model, couldn't handle the gradient imbalance from 93% negatives — the asymmetric loss signal pushed its decision hyperplane somewhere suboptimal, and downsampling had to compensate. LGBM, being more expressive and stopped by AP rather than total loss, found the right boundary from the full distribution without needing to artificially balance the classes.

**The conclusion isn't "always train on everything" or "always downsample." It's: train on everything if your model is expressive enough to use it. If it isn't, downsampling trades data volume for gradient balance — and sometimes that trade is worth it.**